<a href="https://colab.research.google.com/github/fyzdurak/turkish-movie-sentiment-analysis/blob/main/nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import numpy as np
import re
import nltk

# Türkçe stop-words (ve, veya, ama gibi etkisiz kelimeler) için indiriyoruz
nltk.download('stopwords')
from nltk.corpus import stopwords

print("Kütüphaneler başarıyla yüklendi!")
# Veriyi okuyoruz. Türkçe karakter hatası almamak için encoding='utf-8' ekledik.
df = pd.read_csv('/content/turkish_movie_sentiment_dataset.csv')

# Veri setinin büyüklüğü(Kaç satır, kaç sütun?)
print("Veri Seti Boyutu:", df.shape)

# İlk 5
df.head()
turkce_stopwords = set(stopwords.words('turkish'))

def veri_temizle(metin):
    # 1. Metni küçük harfe çeviriyoruz
    metin = str(metin).lower()

    # 2. Sayıları, noktalama işaretlerini ve web linklerini temizliyoruz
    metin = re.sub(r'http\S+|www\S+|https\S+', '', metin, flags=re.MULTILINE) # Linkleri siler
    metin = re.sub(r'[^a-zA-ZğüşıöçĞÜŞİÖÇ\s]', '', metin) # Sadece harfleri bırakır

    # 3. Kelimeleri ayırıyoruz (Tokenization)
    kelimeler = metin.split()

    # 4. Etkisiz kelimeleri (ve, veya, ama, çünkü...) listeden atıyoruz
    temiz_kelimeler = [kelime for kelime in kelimeler if kelime not in turkce_stopwords]

    # 5. Temizlenen kelimeleri tekrar birleştirip cümle yapıyoruz
    return " ".join(temiz_kelimeler)

# Fonksiyonu test edelim:
test = "Bu film GERÇEKTEN çok kötüydü, zaman kaybı! Asla izlemeyin 2/10."
print("Temizlenmiş Test Hali:", veri_temizle(test))

df['temiz_yorum'] = df['comment'].apply(veri_temizle)
df[['comment', 'temiz_yorum']].head()

df.head()

print("Puan Dağılımı:")
print(df['point'].value_counts().sort_index())

# puanları 0 ve 1'e dönüştüren bir fonksiyon
#  3'ün altını 0, 4 ve 5'i 1

def puani_etikete_cevir(puan):
    # Eğer puan metin olarak geldiyse sayıya çevirelim (örn: "8,3" veya "8.3")
    try:
        puan = float(str(puan).replace(',', '.'))
    except:
        return np.nan # Hatalı bir değer varsa boş bırak

    if puan <= 3.0:
        return 0  # Olumsuz
    else:
        return 1  # Olumlu

# Fonksiyonu uygulayıp 'label' adında yeni bir sütun oluşturalım
df['label'] = df['point'].apply(puani_etikete_cevir)

# Boş kalan (dönüştürülemeyen) satırlar varsa onları silelim
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

# Yeni dağılıma bakalım (Kaç olumlu, kaç olumsuz oldu?)
print("\nYeni Etiket Dağılımı (0 = Olumsuz, 1 = Olumlu):")
print(df['label'].value_counts())

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Verimizi Eğitim (Train) ve Test olarak ikiye bölüyoruz (%80 Eğitim, %20 Test)
X_train, X_test, y_train, y_test = train_test_split(
    df['temiz_yorum'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label'] # Olumlu/olumsuz oranını korumak için
)

# 2. TF-IDF Dönüştürücüsünü hazırlıyoruz
# En sık geçen 5000 kelimeyi alması projenin hızlı ve verimli çalışması için idealdir
tfidf = TfidfVectorizer(max_features=5000)

# 3. Eğitim verisini öğren ve dönüştür
X_train_tfidf = tfidf.fit_transform(X_train)

# 4. Test verisini sadece dönüştür (Eğitimde öğrendiği kelimelere göre)
X_test_tfidf = tfidf.transform(X_test)

print("Eğitim seti boyutu:", X_train_tfidf.shape)
print("Test seti boyutu:", X_test_tfidf.shape)
print("Kelimeler başarıyla sayılara dönüştürüldü!")

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Modelimizi tanımlıyoruz
model = LogisticRegression(max_iter=1000)

# 2. MODELİ EĞİTİYORUZ (Yapay zeka burada TF-IDF sayıları ile 0-1 etiketleri arasındaki ilişkiyi öğreniyor)
model.fit(X_train_tfidf, y_train)

# 3. Modelin hiç görmediği TEST verisi üzerinde tahmin yapmasını istiyoruz
y_pred = model.predict(X_test_tfidf)

# 4. SONUÇLARI DEĞERLENDİRİYORUZ
print("--- MODEL BAŞARI RAPORU ---")
print(f"Genel Doğruluk (Accuracy) Skoru: %{accuracy_score(y_test, y_pred) * 100:.2f}")
print("\nDetaylı Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred))

def duygu_tahmin_et(cumle):
    # 1. Cümleyi önce temizliyoruz
    temiz_cumle = veri_temizle(cumle)

    # 2. Kelimeleri TF-IDF matrisine dönüştürüyoruz (Eğittiğimiz tfidf nesnesini kullanarak)
    vektorize_cumle = tfidf.transform([temiz_cumle])

    # 3. Modele tahmin ettiriyoruz
    tahmin = model.predict(vektorize_cumle)[0]
    olasiliklar = model.predict_proba(vektorize_cumle)[0] # Ne kadar emin olduğunu görmek için

    # 4. Sonucu ekrana yazdırıyoruz
    if tahmin == 1:
        durum = "😊 OLUMLU"
        eminlik = olasiliklar[1] * 100
    else:
        durum = "😞 OLUMSUZ"
        eminlik = olasiliklar[0] * 100

    print(f"Cümle: '{cumle}'")
    print(f"Tahmin: {durum} (Eminlik Oranı: %{eminlik:.2f})\n")

# ---- TEST EDELİM ----

duygu_tahmin_et("Hayatımda izlediğim en muhteşem filmdi, oyuncu kadrosu harika.")
duygu_tahmin_et("Tam bir zaman kaybı, param boşa gitti. Kimseye tavsiye etmem.")
duygu_tahmin_et("Görsel efektler idare eder ama senaryo çok kopuk ve sıkıcıydı.")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Kütüphaneler başarıyla yüklendi!
Veri Seti Boyutu: (83227, 3)
Temizlenmiş Test Hali: film gerçekten kötüydü zaman kaybı asla izlemeyin
Puan Dağılımı:
point
0,5     5150
1,0     4917
1,5     1464
2,0     2433
2,5    11433
3,0     5499
3,1       26
3,2       38
3,5     8565
3,7       55
3,8      100
3,9      211
4,0    19958
4,5     6983
4,6      522
5,0    15873
Name: count, dtype: int64

Yeni Etiket Dağılımı (0 = Olumsuz, 1 = Olumlu):
label
1    52331
0    30896
Name: count, dtype: int64
Eğitim seti boyutu: (66581, 5000)
Test seti boyutu: (16646, 5000)
Kelimeler başarıyla sayılara dönüştürüldü!
--- MODEL BAŞARI RAPORU ---
Genel Doğruluk (Accuracy) Skoru: %77.69

Detaylı Sınıflandırma Raporu:
              precision    recall  f1-score   support

           0       0.75      0.60      0.66      6179
           1       0.79      0.88      0.83     10467

    accuracy                           0.78     16646
   macro avg       0.77      0.74      0.75     16646
weighted avg       0.77    